[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DucThinhmta/MentalHealthAI/blob/main/notebooks/train_and_evaluate_colab.ipynb)

# MentalHealthAI: huấn luyện lại và đo trên GPU (Bước 1-2)

Chọn `Runtime > Change runtime type > GPU (T4)`, rồi chạy **từ trên xuống**. Phiên Colab có thể bị ngắt giữa chừng, nên mọi kết quả được **sao lưu lên Drive**
sau mỗi giai đoạn và **khôi phục tự động** khi chạy lại; cờ `--skip-done` bỏ qua các seed đã xong, nên bị ngắt chỉ mất phần đang chạy dở.

| Phần | Làm gì | Ước lượng (T4; chỉ là ước lượng, phụ thuộc cỡ dữ liệu) |
|---|---|---|
| 0 | Lấy mã, cài thư viện, khôi phục kết quả cũ | vài phút |
| 1 | Dữ liệu, làm sạch, baseline TF-IDF | vài phút |
| 2 | **Bước 1:** đo lại 3 mô hình hiện có | 10-20 phút |
| 3 | **Bước 2:** Depression nhiều seed, so với baseline | 1-3 giờ |
| 4 | Stress nhiều seed (tuỳ chọn) | 1-2 giờ |

**Quy tắc:** tập test chỉ để báo cáo, không dùng để chọn mô hình hay ngưỡng. Báo cáo trung bình ± độ lệch của nhiều seed; nếu BERT không vượt baseline thì ghi đúng như vậy.

## 0. Chuẩn bị (ô nào cũng chạy trên máy chủ Google, không tải gì về máy bạn)

**Colab Secrets** (biểu tượng chìa khoá ở thanh bên trái, bật "Notebook access" cho từng khoá): dùng để cất khoá thay vì dán vào ô mã.
- `HF_TOKEN`: chỉ cần nếu 3 mô hình trên Hugging Face ở chế độ private.
- `GITHUB_TOKEN`: chỉ cần nếu repo GitHub ở chế độ private (quyền đọc).
- `ANTHROPIC_API_KEY`: tuỳ chọn, bật chế độ LLM (trò chuyện tự nhiên và dịch). Không có thì bot trả lời theo luật.

**Không bao giờ dán khoá vào ô mã hoặc chia sẻ notebook đã chứa khoá.**

In [ ]:
import subprocess
import torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > GPU"

In [ ]:
import os, subprocess, sys
REPO = "DucThinhmta/MentalHealthAI"
BRANCH = "main"            # đổi sang nhánh khác nếu muốn thử tính năng chưa gộp
sys.path.insert(0, "/content/MentalHealthAI/src")
if not os.path.exists("/content/MentalHealthAI"):
    tok = ""
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass
    url = f"https://{tok + '@' if tok else ''}github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/MentalHealthAI"], check=True)
%cd /content/MentalHealthAI
import colab_helpers as ch
os.environ["HF_TOKEN"] = ch.secret("HF_TOKEN", "") or ""
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install -r requirements-colab.txt
from google.colab import drive
drive.mount("/content/drive")
import pathlib
DRIVE = pathlib.Path("/content/drive/MyDrive/MentalHealthAI_colab")     # thư mục chung: chia sẻ cho cả nhóm (quyền chỉnh sửa)
DRIVE.mkdir(parents=True, exist_ok=True)
SYNC = [("reports", DRIVE / "reports"), ("models/seeds", DRIVE / "seeds_light"), ("models/calibration.json", None)]
SYNC = [(l, d) for l, d in SYNC if d]
def restore(): print("khôi phục:", ch.sync_dirs(SYNC, "from_drive"), "tệp")
def backup():  print("sao lưu:", ch.sync_dirs(SYNC, "to_drive"), "tệp")
restore()

## 1. Dữ liệu

Repo **không chứa dữ liệu** (điều khoản của từng bộ khác nhau). **Một người** (người giữ dữ liệu) đưa các tệp vào thư mục Drive `MentalHealthAI_colab/data/` **một lần** và chia sẻ thư mục `MentalHealthAI_colab` cho cả nhóm
(Drive > chuột phải > Share > thêm email thành viên, quyền Editor). Các thành viên khác mở Drive của mình, chuột phải thư mục được chia sẻ > **Organize > Add shortcut**, rồi chạy tiếp.
Không đặt dữ liệu ở chế độ "anyone with the link". Tên tệp cần có:

- Stress (Dreaddit): `train_clean.csv`, `test_clean.csv`
- Emotion (GoEmotions): `train_emotion.csv`, `dev_emotion.csv`, `test_emotion.csv`
- Depression: `depression_train.csv`, `depression_test.csv`

Bộ dữ liệu bổ sung (UIT-VSMEC...) có thể tải ngay trong Colab bằng `!python src/fetch_datasets.py` (không cần qua Drive).

In [ ]:
import shutil
DATA = DRIVE / "data"
NEEDED = ["train_clean.csv", "test_clean.csv", "train_emotion.csv", "dev_emotion.csv", "test_emotion.csv",
          "depression_train.csv", "depression_test.csv"]
pathlib.Path("data").mkdir(exist_ok=True)
missing = [n for n in NEEDED if not (DATA / n).exists()]
for n in NEEDED:
    if n not in missing:
        shutil.copy(DATA / n, pathlib.Path("data") / n)
print("Thiếu:", missing or "không thiếu tệp nào")
assert not missing, f"Đưa các tệp còn thiếu vào {DATA}"

In [ ]:
!python src/clean_data.py
!python src/baseline_tfidf.py
!cat reports/baselines.csv
backup()

## 2. Bước 1: đo lại 3 mô hình hiện có trên tập test

In [ ]:
!python src/download_models.py
!python src/evaluate_all.py --task all
!python src/calibrate.py
backup()

In [ ]:
import json
print(json.dumps(json.load(open("reports/results.json")), indent=1, ensure_ascii=False)[:3000])

## 3. Bước 2: sửa nhánh Depression (BERT đang thua TF-IDF về macro-F1)

Hai cấu hình × 3 seed: `base` (đối chứng) và `focal` (focal loss, làm dịu trọng số lớp, label smoothing, chọn epoch theo QWK).
Mỗi seed xong là sao lưu. Bị ngắt: chạy lại ô này, các seed đã xong tự được bỏ qua. Hết bộ nhớ GPU: thêm `--batch-size 8` sau dấu `--`.

In [ ]:
SEEDS = "42 43 44"
!python src/run_seeds.py --task depression --tag base  --seeds $SEEDS --skip-done -- --epochs 4 --patience 2
backup()
!python src/run_seeds.py --task depression --tag focal --seeds $SEEDS --skip-done -- --loss focal --weight-power 0.5 --label-smoothing 0.05 --metric qwk --epochs 6 --patience 2
backup()

In [ ]:
!python src/run_seeds.py --compare focal base --task depression --seeds $SEEDS
!python src/compare_depression.py

**Đọc kết quả** (quy tắc đặt trước trong `src/compare_depression.py`): *BERT vượt baseline* → dùng cấu hình đó và ghi khoảng tin cậy vào `docs/MODEL_CARD.md`;
*baseline vượt BERT* → dùng TF-IDF + LogReg cho Depression (hoặc ghép hai mô hình) và ghi đúng như vậy; *chưa phân biệt được* → tăng lên 5-10 seed hoặc báo cáo cả hai.

## 4. (Tuỳ chọn) Stress nhiều seed

In [ ]:
!python src/run_seeds.py --task stress --tag base --seeds $SEEDS --skip-done -- --epochs 4 --patience 2
backup()

## 5. Kết quả nằm ở đâu
Toàn bộ `reports/` và danh sách seed đã được sao lưu vào `MyDrive/MentalHealthAI_colab/` (cả nhóm thấy chung, không phải tải về máy).
Trọng số mô hình nặng **không** được sao lưu mặc định; nếu cần giữ mô hình tốt nhất, chép riêng một thư mục `models/seeds/<tên>` vào Drive.
**Không đẩy dữ liệu lên Git.**